In [11]:
import os
import pandas as pd
import numpy as np

# Rutas
dir_entrada = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW"
dir_salida = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN"
ruta_umbral = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW_filtrMIN\porcentaje_de_datos_min_a_usar.csv"

# Crear carpeta de salida si no existe
os.makedirs(dir_salida, exist_ok=True)

# Leer tabla de umbrales
df_umbral = pd.read_csv(ruta_umbral, sep=";")
df_umbral.columns = ["NOMBRE", "NDVI_total", "NDATOS_min_NDVI", "NDWI_v_total", "NDATOS_min_NDWI_v"]

# Crear diccionarios de umbrales
umbrales_ndvi = dict(zip(df_umbral["NOMBRE"], df_umbral["NDATOS_min_NDVI"]))
umbrales_ndwiv = dict(zip(df_umbral["NOMBRE"], df_umbral["NDATOS_min_NDWI_v"]))

# Procesar cada archivo CSV
for archivo in os.listdir(dir_entrada):
    if archivo.endswith("_sin_periodo_nieve.csv"):
        try:
            ruta_csv = os.path.join(dir_entrada, archivo)
            df = pd.read_csv(ruta_csv, sep=";", encoding="latin1")

            # Nombre de la turbera
            nombre_turbera = archivo.replace("_sin_periodo_nieve.csv", "").replace("_", " ")

            # Intentar formato ISO primero
            fechas = pd.to_datetime(df['FECHA'], errors='coerce')
            if fechas.isna().sum() > 0:
                # Reintentar con dayfirst=True si hubo errores
                fechas_alt = pd.to_datetime(df['FECHA'], dayfirst=True, errors='coerce')
                if fechas_alt.notna().sum() > fechas.notna().sum():
                    fechas = fechas_alt

            # Asignar fechas válidas
            df['FECHA'] = fechas
            errores_fecha = df['FECHA'].isna().sum()
            if errores_fecha > 0:
                print(f"⚠️ {errores_fecha} fechas inválidas en {archivo}")

            # Convertir a formato ISO
            df['FECHA'] = df['FECHA'].dt.strftime('%Y-%m-%d')

            # Aplicar filtros por umbrales
            umbral_ndvi = umbrales_ndvi.get(nombre_turbera)
            umbral_ndwiv = umbrales_ndwiv.get(nombre_turbera)

            if umbral_ndvi is None or umbral_ndwiv is None:
                print(f"⚠️ Umbral no encontrado para {nombre_turbera}")
                continue

            if "NDATOS" in df.columns:
                df.loc[df["NDATOS"] < umbral_ndvi, "NDVI"] = np.nan

            if "NDATOS.3" in df.columns:
                df.loc[df["NDATOS.3"] < umbral_ndwiv, "NDWI_v"] = np.nan

            # Guardar archivo filtrado y corregido
            salida = archivo.replace("_sin_periodo_nieve.csv", "_sin_periodo_nieve_filtr.csv")
            df.to_csv(os.path.join(dir_salida, salida), sep=";", index=False, encoding="latin1")

            print(f"✅ Procesado correctamente: {archivo}")

        except Exception as e:
            print(f"❌ Error con {archivo}: {e}")

print("\n🎯 Proceso finalizado.")


✅ Procesado correctamente: Bassa_Nera_sin_periodo_nieve.csv
✅ Procesado correctamente: Clots_de_Rialba_sin_periodo_nieve.csv
✅ Procesado correctamente: Estanyeres_sin_periodo_nieve.csv
✅ Procesado correctamente: Plan_deth_Horn_sin_periodo_nieve.csv
✅ Procesado correctamente: Pla_de_Sartari_sin_periodo_nieve.csv
✅ Procesado correctamente: Pleta_de_Molieres_sin_periodo_nieve.csv

🎯 Proceso finalizado.
